In [29]:
!sudo apt-get update -qq
!sudo apt-get install -y -qq --no-install-recommends openjdk-17-jre-headless
%pip install -q pyspark sympy

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os, sys, subprocess
os.environ["PYSPARK_PYTHON"] = sys.executable

print(subprocess.run(["java", "-version"], capture_output=True, text=True).stderr)

from pyspark.sql import SparkSession

spark = (SparkSession.builder
         .appName("lab1")
         .master("local[*]")
         .config("spark.driver.memory", "4g")
         .config("spark.sql.shuffle.partitions", "8")
         .getOrCreate())
sc = spark.sparkContext

print(spark.version, sc.parallelize(range(10)).sum())

openjdk version "17.0.20.1" 2026-08-18
OpenJDK Runtime Environment (build 17.0.20.1+1-1-26.04-Ubuntu)
OpenJDK 64-Bit Server VM (build 17.0.20.1+1-1-26.04-Ubuntu, mixed mode, sharing)



Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/07 18:24:06 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
[Stage 0:>                                                        (0 + 14) / 14]

4.3.0 45


In [25]:
import zlib

class Shingling:
    def __init__(self, shingle_length: int):
        self.k = shingle_length

    def construct_k_shingles(self, text: str) -> list[int]:
        index = 0
        doc_length = len(text)
        k_shingles = set()

        while index + self.k <= doc_length:
            shingle = text[index:index + self.k]
            hashed_shingle = self._hash_shingle(shingle)
            k_shingles.add(hashed_shingle)
            index += 1

        return sorted(k_shingles)

    def _hash_shingle(self, s: str) -> int:
        return zlib.crc32(s.encode("utf-8")) 

    def normalize(self, text: str) -> str:
        return " ".join(text.lower().split())

In [55]:
import os

sh = Shingling(shingle_length=10)
dir_path = os.path.abspath("../datasets/sport_documents")

docs = sc.wholeTextFiles(f"file://{dir_path}/*.txt", minPartitions=8)

normalized = docs.mapValues(sh.normalize)
shingled = normalized.mapValues(sh.construct_k_shingles).filter(lambda kv: len(kv[1]) > 0)

shingled.cache()
print(shingled.count(), "documents")

path, hashes = shingled.first()
print(path, len(hashes), hashes[:5])

1000 documents
file:/home/jovyan/work/datasets/sport_documents/Text0001.txt 529 [2170615, 11575593, 14391654, 15309619, 19598772]


In [27]:
class CompareSets:
    def compute_jaccard_similarity(self, shingle_list1: list[int], shingle_list2: list[int]) -> float:
        shingle_set1 = set(shingle_list1)
        shingle_set2 = set(shingle_list2)
        
        intersection_cardinality = len(shingle_set1 & shingle_set2)
        union_cardinality = len(shingle_set1 | shingle_set2)
        
        return intersection_cardinality / union_cardinality

In [47]:
def doc_key(name):
    return f"file:{dir_path}/{name}"

key1 = doc_key("Text0008.txt")
key2 = doc_key("Text0158.txt")

cs = CompareSets()

a = shingled.lookup(key1)
b = shingled.lookup(key2)

sim = cs.compute_jaccard_similarity(a[0], b[0])
print(f"{key1.split('/')[-1]} vs {key2.split('/')[-1]}: {sim:.4f}")
print(len(a[0]), len(b[0]))

Text0008.txt vs Text0158.txt: 0.0010
2679 1491


In [60]:
from sympy import nextprime
from random import Random

class MinHashing:
    def __init__(self, signature_size: int, seed: int = 42):
        self.n = signature_size
        self.c = int(nextprime(2**32 - 1))

        rng = Random(seed)

        self.a = [rng.randint(1, self.c - 1) for _ in range(self.n)]
        self.b = [rng.randint(0, self.c - 1) for _ in range(self.n)]

    def build_minhash_signature(self, hashed_shingles: list[int]) -> list[int]:
        if not hashed_shingles:
            return [self.c] * self.n
    
        return [
            min((a * s + b) % self.c for s in hashed_shingles)
            for a, b in zip(self.a, self.b)
        ]

In [78]:
mh = MinHashing(signature_size=100)
minhashed = shingled.mapValues(mh.build_minhash_signature)

minhashed.cache()
print(minhashed.count(), "documents")

key = doc_key("Text0918.txt")
sig = minhashed.lookup(key)[0]
print(sig)

[Stage 68:==================================================>       (7 + 1) / 8]

1000 documents
[270017, 311726, 51272, 13559, 76783, 112872, 655546, 35755, 7933, 351191, 36675, 73485, 91394, 132101, 337882, 332329, 13351, 653797, 23589, 94468, 33606, 103962, 410276, 424127, 624460, 768639, 12577, 64575, 167788, 320102, 57268, 289100, 379435, 92996, 59129, 838954, 192442, 219554, 414464, 74130, 9615, 181012, 327820, 124128, 152484, 533178, 243496, 99938, 624429, 561402, 115103, 31519, 239676, 128614, 152419, 37076, 421363, 220570, 142508, 250074, 37342, 216070, 278525, 286166, 74571, 632, 243214, 9687, 442075, 83672, 314348, 22372, 33975, 176039, 831607, 184442, 55064, 150043, 104900, 267478, 3684, 173423, 125120, 102544, 100182, 163384, 213255, 92046, 125647, 571174, 77674, 112958, 87141, 601274, 209850, 129010, 191999, 907686, 18839, 102172, 178168, 128893, 105248, 512533, 103324, 264543, 128133, 160495, 41772, 164062, 108148, 136335, 290232, 134121, 84112, 216952, 356220, 617140, 220278, 135207, 14831, 681490, 531209, 279858, 28089, 273663, 61985, 63110, 123080,

In [ ]:
class CompareSignatures:
    def evaluate_signature_similarity(self, signature1: list[int], signature2: list[int]) -> float:
        if len(signature1) != len(signature2):
            raise ValueError(
                f"signature lengths differ: {len(signature1)} vs {len(signature2)}"
            )
        return sum(x == y for x, y in zip(signature1, signature2)) / len(signature1)

In [79]:
cs = CompareSets()
csig = CompareSignatures()

key1 = doc_key("Text0918.txt")
key2 = doc_key("Text0326.txt")

sig1 = minhashed.lookup(key1)
sig2 = minhashed.lookup(key2)
sig1, sig2 = sig1[0], sig2[0]

est = csig.evaluate_signature_similarity(sig1, sig2)

set1 = shingled.lookup(key1)[0]
set2 = shingled.lookup(key2)[0]
true = cs.compute_jaccard_similarity(set1, set2)

print(f"estimated (MinHash, n={len(sig1)}): {est:.4f}")
print(f"true Jaccard:                       {true:.4f}")
print(f"abs error:                          {abs(est - true):.4f}")

estimated (MinHash, n=200): 0.0000
true Jaccard:                       0.0090
abs error:                          0.0090
